In [15]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

from src.evaluation import ndcg_at_k, precision_at_k, recall_at_k
from src.item_based_cf import ItemBasedCF


In [16]:
input_dir = Path("../data")

train = pd.read_csv(input_dir / "processed/train.csv")
test = pd.read_csv(input_dir / "processed/test.csv")
movies = pd.read_csv(input_dir / "ml-latest-small/movies.csv")

In [17]:
user_item = pd.pivot_table(data=train,
                            values="rating",
                            index="userId",
                            columns="movieId",
                            fill_value=0,
                            margins=False
                           )

item_sim_matrix = cosine_similarity(user_item.T)

item_sim = pd.DataFrame(
    item_sim_matrix,
    index = user_item.columns,
    columns = user_item.columns
)

In [18]:
similar_items = item_sim[1].drop(1).sort_values(ascending=False).head(10)
titles = movies.set_index("movieId").loc[similar_items.index]["title"].tolist()
titles

['Toy Story 2 (1999)',
 'Jurassic Park (1993)',
 'Independence Day (a.k.a. ID4) (1996)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Forrest Gump (1994)',
 'Star Wars: Episode VI - Return of the Jedi (1983)',
 'Groundhog Day (1993)',
 'Lion King, The (1994)',
 'Shrek (2001)',
 'Back to the Future (1985)']

In [19]:
userId = 1

user_means = train.groupby('userId')['rating'].mean()

train_centered = train.copy()
train_centered['rating_centered'] = train_centered['rating'] - \
train_centered['userId'].map(user_means)

user_item_centered = train_centered.pivot(index='userId', columns='movieId', 
                                          values='rating_centered').fillna(0)

item_sim_matrix = cosine_similarity(user_item_centered.T)
item_sim = pd.DataFrame(item_sim_matrix, index=user_item_centered.columns, 
                        columns=user_item_centered.columns)

user_vector = user_item.loc[userId]
unwatched = user_vector[user_vector == 0].index.tolist()

def rating(user_item: pd.DataFrame, sim: pd.DataFrame, userId: int, i: int, k: int, 
           min_neighbors: int = 5) -> float:

    watched_mask = user_item.loc[userId] !=  0
    sim_scores = sim.loc[i, watched_mask]
    sim_scores_pos = sim_scores[sim_scores > 0]
    sim_top_k = sim_scores_pos.sort_values(ascending=False).head(k)
    user_ratings = user_item.loc[userId, sim_top_k.index]

    sim_sum = sim_top_k.sum()
    if sim_sum == 0 or len(sim_top_k) < min_neighbors:
        return 0.0, 0

    rating_diff = np.dot(sim_top_k, user_ratings) / sim_sum
    final_rating = user_means.loc[userId] + rating_diff
    final_rating = np.clip(final_rating, 0.5, 5.0)

    return final_rating, len(sim_top_k)

r = [(x, rating(user_item_centered, item_sim, userId, x, 50)) for x in unwatched]    

In [20]:
predict = sorted([x for x in r if x[1][0] > 0], key=lambda x: x[1][0], reverse=True)[:10]
titles_predict = [x[0] for x in predict]
titles = movies.set_index("movieId").loc[titles_predict]["title"]
print(titles)
print(predict)

movieId
2585      Lovers of the Arctic Circle, The (Los Amantes ...
2695                                       Boys, The (1998)
4505                                       For Keeps (1988)
27320                  Nine Lives of Tomas Katz, The (2000)
141718                                     Deathgasm (2015)
167732                        A Street Cat Named Bob (2016)
5919                                         Android (1982)
5181                                       Hangar 18 (1980)
7134      Element of Crime, The (Forbrydelsens Element) ...
77266                                       Disgrace (2008)
Name: title, dtype: str
[(2585, (np.float64(5.0), 8)), (2695, (np.float64(5.0), 8)), (4505, (np.float64(5.0), 8)), (27320, (np.float64(5.0), 8)), (141718, (np.float64(5.0), 5)), (167732, (np.float64(5.0), 5)), (5919, (np.float64(4.990635732301211), 9)), (5181, (np.float64(4.975541442209964), 46)), (7134, (np.float64(4.9743983454044916), 14)), (77266, (np.float64(4.9743983454044916), 14))]


In [21]:
real_top = user_item.loc[userId].sort_values(ascending=False).head(10)
real_movies = (
    movies.set_index("movieId")
    .loc[real_top.index, ["title", "genres"]]
    .assign(user_rating=real_top.values)
)
print(real_movies)

                                             title  \
movieId                                              
260      Star Wars: Episode IV - A New Hope (1977)   
2395                               Rushmore (1998)   
2470                       Crocodile Dundee (1986)   
1954                                  Rocky (1976)   
2872                              Excalibur (1981)   
2058                        Negotiator, The (1998)   
3729                                  Shaft (1971)   
1927         All Quiet on the Western Front (1930)   
3034                             Robin Hood (1973)   
1049            Ghost and the Darkness, The (1996)   

                                              genres  user_rating  
movieId                                                            
260                          Action|Adventure|Sci-Fi          5.0  
2395                                    Comedy|Drama          5.0  
2470                                Adventure|Comedy          5.0  
1954       

In [22]:
i = 43930  # Just My Luck

watched_mask = user_item.loc[userId] > 0
sim_scores = item_sim.loc[i, watched_mask]
sim_scores_pos = sim_scores[sim_scores > 0]
sim_top_k = sim_scores_pos.sort_values(ascending=False).head(50)

print(sim_top_k)
titles = movies.set_index("movieId").loc[sim_top_k.index]["title"].tolist()
print(titles)
user_ratings_for_neighbors = user_item.loc[userId, sim_top_k.index]
print(user_ratings_for_neighbors)

movieId
1197    0.025362
Name: 43930, dtype: float64
['Princess Bride, The (1987)']
movieId
1197    5.0
Name: 1, dtype: float64


In [23]:
model = ItemBasedCF(train, min_neighbors=15)

k = 10

test_grouped = (
    test.groupby("userId")["movieId"]
    .apply(lambda ids: set(ids))
    .reset_index(name="relevant")
)

def evaluate_row(row, model, k):
    try:
        recs = model.recommend_top_n(row["userId"], k)
        recs_list = recs["movieId"].tolist()
    except ValueError:
        recs_list = []

    p = precision_at_k(recs_list, row["relevant"], k)
    r = recall_at_k(recs_list, row["relevant"], k)
    ndcg = ndcg_at_k(recs_list, row["relevant"], k)
    
    return pd.Series([p, r, ndcg], index=[f"precision@{k}", f"recall@{k}", f"ndcg@{k}"])


metrics_df = test_grouped.apply(lambda row: evaluate_row(row, model, k), axis=1)
results_df = pd.concat([test_grouped, metrics_df], axis=1)

print(f"Mean Precision@{k}: {results_df[f'precision@{k}'].mean():.4f}")
print(f"Mean Recall@{k}:    {results_df[f'recall@{k}'].mean():.4f}")
print(f"Mean nDCG@{k}:      {results_df[f'ndcg@{k}'].mean():.4f}")

Mean Precision@10: 0.0002
Mean Recall@10:    0.0016
Mean nDCG@10:      0.0005


In [24]:
lengths = [len(model.recommend_top_n(uid, 10)) for uid in test_grouped["userId"].head(50)]
print(pd.Series(lengths).describe())

count    50.0
mean     10.0
std       0.0
min      10.0
25%      10.0
50%      10.0
75%      10.0
max      10.0
dtype: float64


In [25]:
test_items = set(test["movieId"].unique())
train_items = set(model.item_sim_.columns)
print(len(test_items - train_items), "из", len(test_items), "тестовых фильмов отсутствуют в train")

22 из 510 тестовых фильмов отсутствуют в train


In [26]:
uid = test_grouped["userId"].iloc[0]
relevant_movie = list(test_grouped["relevant"].iloc[0])[0]
preds = model._predict_rating(uid)
print(relevant_movie in preds.index, preds.get(relevant_movie, "не найден"))

True 4.064020732221454


In [27]:
uid = test_grouped["userId"].iloc[0]
relevant_movie = list(test_grouped["relevant"].iloc[0])[0]

preds = model._predict_rating(uid)
preds_sorted = preds.sort_values(ascending=False)

rank = preds_sorted.index.get_loc(relevant_movie)
print(f"Ранг relevant-фильма: {rank} из {len(preds_sorted)}")
print(preds_sorted.head(10))

Ранг relevant-фильма: 6921 из 9471
movieId
5181     4.975541
3453     4.946766
4154     4.892677
4031     4.886469
3410     4.880378
341      4.851613
33312    4.838938
3325     4.837155
8593     4.828518
4759     4.828518
dtype: float64


In [28]:
results = []
for min_neighbors in [3, 5, 10]:
    for beta in [10, 20, 30]:
        model = ItemBasedCF(train, min_neighbors=min_neighbors, beta=beta)  # добавь beta как параметр конструктора
        # прогони полную оценку на всех 610 юзерах (как раньше)
        results.append({"min_neighbors": min_neighbors, "beta": beta, 
                         "precision": mean_precision, "recall": mean_recall, "ndcg": mean_ndcg})

pd.DataFrame(results).sort_values("ndcg", ascending=False)

TypeError: ItemBasedCF.__init__() got an unexpected keyword argument 'beta'